# 1.3 数据配比 (Data Mixture)

数据配比决定了模型在不同领域能力的均衡性，是预训练效果的关键因素之一。

本节涵盖：
- 领域配比优化
- 课程学习
- 数据重采样

## 1. 领域配比优化

**目的**：确定不同领域数据的最佳混合比例

**基本原理**：通过网格搜索、DoReMi等方法自动搜索最优数据混合比例，使模型在各领域上获得均衡能力，避免某领域数据过多导致的能力偏斜。

**DoReMi方法**：
1. 先用均匀混合的数据训练一个小的参考模型
2. 参考模型在各领域数据上计算困惑度
3. 根据困惑度调整领域权重：困惑度高的领域获得更高权重（模型在该领域还有很大提升空间）
4. 使用调整后的权重重新采样数据训练最终模型

**产业实践**：LLaMA使用了约67%的网页数据、4.5%的Books、4.5%的GitHub、2.5%的Wikipedia等配比。

In [ ]:
import numpy as np

np.random.seed(42)

class DomainMixtureOptimizer:
    def __init__(self, domains, initial_weights=None):
        self.domains = domains
        n = len(domains)
        self.weights = np.array(initial_weights) if initial_weights else np.ones(n) / n

    def compute_perplexity(self, domain_losses):
        return {domain: np.exp(loss) for domain, loss in domain_losses.items()}

    def doremi_update(self, domain_losses, lr=0.1):
        perplexities = self.compute_perplexity(domain_losses)
        total_ppl = sum(perplexities.values())
        n_domains = len(self.domains)

        new_weights = np.zeros(n_domains)
        for i, domain in enumerate(self.domains):
            excess_ppl = perplexities[domain] / (total_ppl / n_domains)
            new_weights[i] = np.log(excess_ppl + 1e-8)

        new_weights = self.weights + lr * new_weights
        new_weights = np.maximum(new_weights, 0)
        new_weights /= new_weights.sum()
        self.weights = new_weights
        return self.weights.copy()

    def sample_batch(self, batch_size, domain_data):
        indices = np.random.choice(len(self.domains), size=batch_size, p=self.weights)
        batch = []
        for idx in indices:
            domain = self.domains[idx]
            sample = np.random.choice(domain_data[domain])
            batch.append((domain, sample))
        return batch

domains = ['web', 'books', 'code', 'academic', 'wikipedia']
domain_losses = {
    'web': 3.5,
    'books': 2.8,
    'code': 4.2,
    'academic': 3.9,
    'wikipedia': 2.1,
}

optimizer = DomainMixtureOptimizer(domains)

print('=== Domain Mixture Optimization (DoReMi) ===')
print(f'Initial weights: {dict(zip(domains, optimizer.weights))}')

ppls = optimizer.compute_perplexity(domain_losses)
print(f'\nPerplexity per domain:')
for d in domains:
    print(f'  {d:12s}: loss={domain_losses[d]:.1f}, ppl={ppls[d]:.1f}')

print(f'\n--- DoReMi Optimization ---')
for step in range(5):
    new_weights = optimizer.doremi_update(domain_losses, lr=0.3)
    print(f'Step {step+1}: {dict(zip(domains, np.round(new_weights, 3)))}')

print(f'\nInsight: Domains with higher perplexity (code, academic) get more weight,')
print(f'because the model has more room to improve on those domains.'

## 2. 课程学习

**目的**：按从易到难的顺序组织训练数据

**基本原理**：模拟人类学习过程，先让模型学习简单、通用的数据，再逐步引入困难、专业的数据，提升训练效率和最终性能。

**难度评估方法**：
- 基于损失值：模型在样本上的损失越高，认为越难
- 基于文本长度：短文本通常比长文本简单
- 基于词汇复杂度：词汇多样性越高越难
- 基于领域专业性：通用领域比专业领域简单

In [ ]:
import numpy as np

np.random.seed(42)

class CurriculumScheduler:
    def __init__(self, samples_with_difficulty):
        self.samples = sorted(samples_with_difficulty, key=lambda x: x[1])
        self.n_samples = len(self.samples)

    def get_curriculum_indices(self, progress):
        progress = np.clip(progress, 0.0, 1.0)
        max_difficulty = self.samples[-1][1]
        threshold = progress * max_difficulty
        available = [(i, s, d) for i, (s, d) in enumerate(self.samples) if d <= threshold]
        if not available:
            available = [(0, self.samples[0][0], self.samples[0][1])]
        return available

    def simulate_training(self, n_steps=100):
        losses = []
        for step in range(n_steps):
            progress = step / n_steps
            available = self.get_curriculum_indices(progress)
            avg_difficulty = np.mean([d for _, _, d in available])
            simulated_loss = max(0.1, 2.0 - progress * 1.5 + np.random.normal(0, 0.1))
            losses.append(simulated_loss)
        return losses

samples = []
difficulties = {
    'simple_qa': 0.1, 'basic_math': 0.2, 'reading_comp': 0.3,
    'code_basic': 0.4, 'science_text': 0.5, 'legal_text': 0.6,
    'code_advanced': 0.7, 'math_proof': 0.8, 'research_paper': 0.9,
    'novel_writing': 1.0,
}
for name, diff in difficulties.items():
    for _ in range(10):
        samples.append((name, diff + np.random.normal(0, 0.05)))

scheduler = CurriculumScheduler(samples)

print('=== Curriculum Learning Demo ===')
for progress in [0.0, 0.2, 0.5, 0.8, 1.0]:
    available = scheduler.get_curriculum_indices(progress)
    types = set(s for _, s, _ in available)
    avg_diff = np.mean([d for _, _, d in available])
    print(f'Progress {progress:.0%}: {len(available)} samples, avg_difficulty={avg_diff:.2f}')
    print(f'  Available types: {sorted(types)}')

curriculum_losses = scheduler.simulate_training(100)
random_losses = [max(0.1, 2.0 - i/100*1.2 + np.random.normal(0, 0.15)) for i in range(100)]

print(f'\nSimulated training comparison (100 steps):')
print(f'  Curriculum: final_loss={np.mean(curriculum_losses[-10:]):.3f}')
print(f'  Random:     final_loss={np.mean(random_losses[-10:]):.3f}')
print(f'  Curriculum converges faster and achieves lower final loss.')

## 3. 数据重采样

**目的**：控制高质量数据的出现频率

**基本原理**：对高质量数据源进行过采样（增加出现频率），对低质量但多样化的数据源进行欠采样，平衡质量与多样性。

**重采样策略**：
- **等权重采样**：每个数据源采样概率相同
- **按数据量加权**：大数据源获得更多采样次数
- **质量加权采样**：高质量数据源获得更高采样权重
- **温度采样**：通过温度参数控制分布的平滑程度

In [ ]:
import numpy as np

np.random.seed(42)

class DataResampler:
    def __init__(self, data_sources, quality_scores, sizes):
        self.sources = data_sources
        self.qualities = np.array(quality_scores)
        self.sizes = np.array(sizes, dtype=float)
        self.n_sources = len(data_sources)

    def uniform_sampling(self, n_samples):
        weights = np.ones(self.n_sources) / self.n_sources
        return self._sample(weights, n_samples)

    def size_weighted_sampling(self, n_samples):
        weights = self.sizes / self.sizes.sum()
        return self._sample(weights, n_samples)

    def quality_weighted_sampling(self, n_samples):
        weights = self.qualities / self.qualities.sum()
        return self._sample(weights, n_samples)

    def temperature_sampling(self, n_samples, temperature=1.0):
        log_weights = np.log(self.qualities + 1e-8) / temperature
        log_weights -= log_weights.max()
        weights = np.exp(log_weights)
        weights /= weights.sum()
        return self._sample(weights, n_samples)

    def _sample(self, weights, n_samples):
        indices = np.random.choice(self.n_sources, size=n_samples, p=weights)
        counts = np.bincount(indices, minlength=self.n_sources)
        return dict(zip(self.sources, counts)), weights

sources = ['wikipedia', 'books', 'arxiv', 'github', 'common_crawl']
qualities = [0.95, 0.85, 0.90, 0.80, 0.50]
sizes = [20, 35, 50, 100, 5000]

resampler = DataResampler(sources, qualities, sizes)
n_samples = 10000

print('=== Data Resampling Strategies ===')
print(f'Sources: {sources}')
print(f'Quality: {qualities}')
print(f'Size (GB): {sizes}')

strategies = {
    'Uniform': resampler.uniform_sampling(n_samples),
    'Size-weighted': resampler.size_weighted_sampling(n_samples),
    'Quality-weighted': resampler.quality_weighted_sampling(n_samples),
    'Temp=0.5': resampler.temperature_sampling(n_samples, temperature=0.5),
    'Temp=2.0': resampler.temperature_sampling(n_samples, temperature=2.0),
}

print(f'\n--- Sampling Distribution ({n_samples} samples) ---')
print(f'{"Strategy":18s}', end='')
for s in sources:
    print(f' {s[:8]:>8s}', end='')
print()
print('-' * 70)
for name, (counts, weights) in strategies.items():
    print(f'{name:18s}', end='')
    for s in sources:
        pct = counts[s] / n_samples * 100
        print(f' {pct:7.1f}%', end='')
    print()

print(f'\nKey insight:')
print(f'  Size-weighted: dominated by common_crawl (largest but lowest quality)')
print(f'  Quality-weighted: favors wikipedia/arxiv (high quality)')
print(f'  Temperature: T<1 sharpens distribution, T>1 flattens it')